**Initialize the REST Client**

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient
from getpass import getpass
import dbrepo.api.dto as dto
from dbrepo.api.exceptions import MalformedError, NameExistsError
from dbrepo.api.dto import IdentifierType, CreateIdentifierTitle, CreateIdentifierCreator, CreateIdentifierDescription

# Load shared config from ../.env (endpoint, container id, database id,
# username). See .env.example in the repo root.
load_dotenv("../.env")

DATABASE_ID = os.environ["DBREPO_DATABASE_ID"]
print("DATABASE_ID:", DATABASE_ID)

DATABASE_ID: 10dfb46a-bc0b-47bc-9802-e927a33c5e75


In [2]:
# Read shared configuration from environment (loaded from ../.env above).
DBREPO_ENDPOINT       = os.environ["DBREPO_ENDPOINT"]
CONTAINER_ID          = os.environ["DBREPO_CONTAINER_ID"]
DATABASE_DISPLAY_NAME = os.environ["DBREPO_DATABASE_NAME"]

username = os.environ["DBREPO_USERNAME"]
user_password = getpass(f"Enter password for {username}: ")

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=username,
    password=user_password,
)
print("Authenticated as:", client.whoami())

Enter password for ahmadhabib:  ········


ahmadhabib
Authenticated as: ahmadhabib


In [3]:
containers = client.get_containers()
print(containers)
if len(containers) == 0:
    print("No containers found. Please create a container first.")
if len(containers) == 1:
    old = CONTAINER_ID
    CONTAINER_ID = containers[0].id
    print(f"container_id has been updated {old} --> {CONTAINER_ID}")
else:
    print("More than one container found. Please specify the container_id manually.")

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]
container_id has been updated 6cfb3b8e-1792-4e46-871a-f3d103527203 --> 6cfb3b8e-1792-4e46-871a-f3d103527203


**Create the Traffic Database**

In [4]:
# Create the database if one isn't already configured. The id is read
# from DBREPO_DATABASE_ID in .env — leave it empty to create a fresh
# database under your account, then paste the printed id back into
# .env so subsequent notebooks find the same database.
#
# is_public=True so the data is discoverable and citable; the owner
# (you) still controls writes. Source data is CC-BY, so no privacy
# concern. Grant teammates write access via client.create_database_access().
DATABASE_ID = os.environ.get("DBREPO_DATABASE_ID") or None

if DATABASE_ID is None:
    database = client.create_database(
        name=DATABASE_DISPLAY_NAME,
        container_id=CONTAINER_ID,
        is_public=True,
        is_schema_public=True,
    )
    DATABASE_ID = database.id
    print(
        "Created new database. Paste this id into .env as "
        f"DBREPO_DATABASE_ID={DATABASE_ID}"
    )
else:
    print(f"Using existing database id from .env: {DATABASE_ID}")

Using existing database id from .env: 10dfb46a-bc0b-47bc-9802-e927a33c5e75


In [5]:
# Create the Identifier (Metadata & Citability)
# This is where you specify the original publisher and license to ensure
# semantic correctness and proper attribution.
# This cells fails if the database already exists.

try:
    client.create_identifier(
        database_id=DATABASE_ID,
        type=IdentifierType.DATABASE,
        creators=[
            CreateIdentifierCreator(
                creator_name="Group 5"
            )
        ],
        titles=[
            CreateIdentifierTitle(
            title=DATABASE_DISPLAY_NAME,
            language=dto.Language.EN # Explicitly defined string mapping
            )
        ],
        descriptions=[
            CreateIdentifierDescription(
                description="Relational 3NF representation of SDCC Traffic Flow Data (Jan-June 2022). "
                            "Original data sourced from the SDCC Open Data Portal.",
                language=dto.Language.EN,
                type=dto.DescriptionType.ABSTRACT
            )
        ],
        # Set the original publisher of the traffic data
        publisher="South Dublin County Council",

        publication_year=2022,

        related_identifiers=[
            dto.CreateRelatedIdentifier(
                value="https://data.smartdublin.ie/dataset/traffic-flow-data-jan-to-june-2022-sdcc1",
                type=dto.RelatedIdentifierType.URL,
                relation=dto.RelatedIdentifierRelation.CITES
            )
        ],
        licenses=[
            dto.License(
                identifier="CC-BY",
                uri="http://www.opendefinition.org/licenses/cc-by",
                description=""
            )
        ]
    )

    print("Metadata identifier created successfully. The database is now citable.")
except MalformedError as e:
    print(f"Skipping identifier creation (already exists or malformed): {e}")

Skipping identifier creation (already exists or malformed): Failed to create identifier: {"type":"about:blank","title":"Bad Request","status":400,"detail":"Invalid request content.","instance":"/api/v1/identifier","properties":null}


In [6]:
# check the identifiers
client.get_identifiers(DATABASE_ID)

[]

**Create the Tables and upload the data**

In [7]:
# --- 1. Load the Measurements (CSV) ---
# CSV_DATA_URL = "https://data-sdublincoco.opendata.arcgis.com/api/download/v1/items/ce994c07d66e4ce582c4d608f339fcd9/csv?layers=0"
# df_measurements = pd.read_csv(CSV_DATA_URL, on_bad_lines='warn')
df_raw = pd.read_csv("../data/raw/Traffic_Flow_Data_Jan_to_June_2022_SDCC.csv", on_bad_lines='warn')

print("Number of rows:", len(df_raw))
print("Available columns csv:", df_raw.columns.tolist())
print("Number of unique sites:", len(df_raw["site"].unique()))
print("Number of unique dates:", len(df_raw["date"].unique()))

Number of rows: 1048575
Available columns csv: ['site', 'day', 'date', 'start_time', 'end_time', 'flow', 'flow_pc', 'cong', 'cong_pc', 'dsat', 'dsat_pc', 'ObjectId']
Number of unique sites: 61
Number of unique dates: 181


**CALENDAR (Dimension Table)**

In [8]:
# --- 2. CALENDAR (Dimension Table) ---
df_calendar = df_raw[['date', 'day']].drop_duplicates().reset_index(drop=True)
df_calendar.columns = ['date', 'day_of_week']

# Source CSV stores dates as DD/MM/YYYY. Convert to ISO YYYY-MM-DD so
# DBRepo's auto-detected DATE column accepts the values.
df_calendar['date'] = pd.to_datetime(
    df_calendar['date'], format='%d/%m/%Y'
).dt.strftime('%Y-%m-%d')

df_calendar["day_of_week"] = df_calendar["day_of_week"].astype(str)
df_calendar = df_calendar.set_index("date")

print("\nCalendar Table Sample:\n", df_calendar.head())


Calendar Table Sample:
            day_of_week
date                  
2022-01-04          TU
2022-01-01          SA
2022-01-03          MO
2022-01-02          SU
2022-01-05          WE


In [9]:
print(len(df_raw["date"].unique()), "unique dates found in raw data.")
print(len(df_calendar), "unique dates found.")

181 unique dates found in raw data.
181 unique dates found.


In [10]:
try:
    calendar_table = client.create_table(
        database_id=DATABASE_ID,
        name="Calendar",
        is_public=True,
        is_schema_public=True,
        dataframe=df_calendar,
        description="Table to handle date-to-day mapping.",
        with_data=True,
    )

    print(f"Created table: {calendar_table.name}")
except NameExistsError as e:
    print("Skipping data upload. Data already exists.")

Skipping data upload. Data already exists.


**TRAFFIC_SITES (Dimension Table)**

In [11]:
# --- 2. TRAFFIC_SITES (Dimension Table) ---
df_sites = df_raw[['site']].drop_duplicates().reset_index(drop=True)
df_sites.columns = ['site_id']
df_sites = df_sites.set_index('site_id')

print("Sites Table\n", df_sites.head())
print(df_sites.info())

Sites Table
 Empty DataFrame
Columns: []
Index: [N01111A, N01111C, N01111D, N01111M, N01111N]
<class 'pandas.core.frame.DataFrame'>
Index: 61 entries, N01111A to N03121D
Empty DataFrame
None


In [12]:
try:
    traffic_sites_table = client.create_table(
        database_id=DATABASE_ID,
        name="TrafficSites",
        is_public=True,
        is_schema_public=True,
        dataframe=df_sites,
        description="Contains data for physical sensor locations.",
        with_data=True,
    )

    print(f"Created table: {traffic_sites_table.name}")
except NameExistsError as e:
    print("Skipping data upload. Data already exists.")

Created table: TrafficSites


**TRAFFIC_MEASUREMENTS (Fact Table)**

In [13]:
# --- 4. TRAFFIC_MEASUREMENTS (Fact Table) ---
# Hier speichern wir die Messdaten und verknüpfen sie über Fremdschlüssel (site_id, date)
df_measurements = df_raw[[
    'site', 'date', 'end_time', 'flow', 'flow_pc',
    'cong', 'cong_pc', 'dsat', 'dsat_pc'
]].copy()

# Spaltennamen an das ER-Modell anpassen
df_measurements.rename(columns={'site': 'site_id'}, inplace=True)

# Source CSV stores dates as DD/MM/YYYY. Convert to ISO YYYY-MM-DD so
# DBRepo's auto-detected DATE column accepts the values, and so the
# foreign-key join against Calendar.date works.
df_measurements['date'] = pd.to_datetime(
    df_measurements['date'], format='%d/%m/%Y'
).dt.strftime('%Y-%m-%d')

# Derive start_time from end_time - 15min. '24:00' in end_time is the
# notation used by the source for midnight, so wrap it to '00:00' before
# parsing.
df_measurements['start_time'] = (pd.to_datetime(
    df_measurements['end_time'].str.replace('24:00', '00:00'),
    format='%H:%M'
) - pd.Timedelta(minutes=15)).dt.strftime('%H:%M')

# Force pandas StringDtype on the time columns so DBRepo's schema
# inference treats them as TEXT, not TIMESTAMP. Bare 'HH:MM' values
# would fail MariaDB's DATETIME validation otherwise.
df_measurements['start_time'] = df_measurements['start_time'].astype('string')
df_measurements['end_time']   = df_measurements['end_time'].astype('string')

# Primärschlüssel (observation_id) generieren
df_measurements.insert(0, 'observation_id', range(1, len(df_measurements) + 1))
df_measurements = df_measurements.set_index(['observation_id'])

print("\nMeasurements Table Sample:\n", df_measurements.head())
print(df_measurements.info())


Measurements Table Sample:
                 site_id        date end_time  flow  flow_pc  cong  cong_pc  \
observation_id                                                               
1               N01111A  2022-01-04    03:15    13      100     0      100   
2               N01111A  2022-01-04    03:30    10      100     0      100   
3               N01111A  2022-01-04    03:45     0      100     0      100   
4               N01111A  2022-01-04    04:00     9      100     0      100   
5               N01111A  2022-01-04    04:15     0      100     0      100   

                dsat  dsat_pc start_time  
observation_id                            
1                  0        0      03:00  
2                  0        0      03:15  
3                  0        0      03:30  
4                  0        0      03:45  
5                  0        0      04:00  
<class 'pandas.core.frame.DataFrame'>
Index: 1048575 entries, 1 to 1048575
Data columns (total 10 columns):
 #   Column    

In [18]:
try:
    traffic_measurements_table = client.create_table(
        database_id=DATABASE_ID,
        name="TrafficMeasurements",
        is_public=True,
        is_schema_public=True,
        dataframe=df_measurements,
        description="Core fact table containing 15-minute traffic flow and congestion metrics.",
        with_data=True,
    )

    print(f"Created table: {traffic_measurements_table.name}")
    print("Data upload complete!")
except NameExistsError as e:
    print("Skipping data upload. Data already exists.")

2026-05-24 16:32:57,365 root         WARNING default to 'text' for column end_time and type <class 'numpy.dtype'>
2026-05-24 16:33:00,908 root         WARNING default to 'text' for column start_time and type <class 'numpy.dtype'>
Skipping data upload. Data already exists.


### Task 2.2: Semantic Attribute Mapping
For each traffic measurement attribute, we map the column to a semantically meaningful ontological concept to enable interoperability and machine-readable understanding of the data.

We leverage three complementary ontologies to ensure comprehensive semantic coverage:

- **SSN/SOSA (Semantic Sensor Network)**: `http://www.w3.org/ns/ssn/Property`  
  Used for observable properties like traffic flow and degree of saturation. SSN provides a robust framework for sensor observations.

- **SAREF4AUTO (Smart Appliances Reference for Automotive)**: `https://saref.etsi.org/saref4auto/Lane_traffic`  
  Applied to congestion indicators. This ETSI standard ontology specifically models automotive and traffic-related concepts.

- **QUDT (Quantities, Units, Dimensions and Types)**: `http://qudt.org/vocab/unit/PERCENT`  
  Used for all percentage-based columns (`flow_pc`, `cong_pc`, `dsat_pc`). QUDT provides standardized representations for units of measurement.

This multi-ontology approach ensures that both the **nature of the measurement** (via SSN/SAREF4AUTO) and its **unit of measure** (via QUDT) are semantically annotated, enabling rich queries and integration with other semantic datasets.

In [19]:

# 1. Fetch the exact table details to get the structural Column IDs
tables = client.get_tables(database_id=DATABASE_ID)
traffic_measurements_table = next(
    t for t in tables if t.name == "TrafficMeasurements"
)

table_details = client.get_table(
    database_id=DATABASE_ID,
    table_id=traffic_measurements_table.id
)

# 2. Define semantic ontology URI mappings (T2.2)
semantic_mappings = {
    "flow": "http://www.w3.org/ns/ssn/Property",                    # SSN: Observed property
    "flow_pc": "http://qudt.org/vocab/unit/PERCENT",                # QUDT: Percentage
    "cong": "https://saref.etsi.org/saref4auto/Lane_traffic",       # SAREF4AUTO: Lane traffic
    "cong_pc": "http://qudt.org/vocab/unit/PERCENT",                # QUDT: Percentage
    "dsat": "http://www.w3.org/ns/ssn/Property",                    # SSN: Observed property
    "dsat_pc": "http://qudt.org/vocab/unit/PERCENT"                 # QUDT: Percentage
}

print("Beginning semantic mapping updates (T2.2)...")

# 3. Iterate through columns and update semantic metadata
for column in table_details.columns:
    if column.name in semantic_mappings:
        try:
            client.update_table_column(
                database_id=DATABASE_ID,
                table_id=column.table_id,
                column_id=column.id,
                concept_uri=semantic_mappings[column.name]
            )
            print(f"Mapped column '{column.name}' -> {semantic_mappings[column.name]}")
        except Exception as e:
            print(f"ERROR updating {column.name}: {e}")

print("\nT2.2 complete!")
print("Documentation: docs/semantic_mapping.md")

Beginning semantic mapping updates (T2.2)...
Mapped column 'flow' -> http://www.w3.org/ns/ssn/Property
Mapped column 'flow_pc' -> http://qudt.org/vocab/unit/PERCENT
Mapped column 'cong' -> https://saref.etsi.org/saref4auto/Lane_traffic
Mapped column 'cong_pc' -> http://qudt.org/vocab/unit/PERCENT
Mapped column 'dsat' -> http://www.w3.org/ns/ssn/Property
Mapped column 'dsat_pc' -> http://qudt.org/vocab/unit/PERCENT

T2.2 complete!
Documentation: docs/semantic_mapping.md
